# Extraer facturas y conceptos etiquetados


Este notebook transforma el paquete de facturas anonimizadas en **dos tablas relacionadas y estructuradas**: una que contiene la información de cada factura de origen y otra que reúne los conceptos registrados en sus respectivos detalles.


In [1]:
%pip install -q pandas pyarrow lxml

Note: you may need to restart the kernel to use updated packages.


In [2]:
import io
import re
from decimal import Decimal, InvalidOperation
from hashlib import sha256
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

import pandas as pd
from IPython.display import display
from lxml import etree

## 1. Entrada y salidas


In [3]:
candidates = [
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "ecuador-einvoice-classifier")
]

PROJECT_DIR = next(
    (p for p in candidates if (p / "ml/notebooks/01_data_extractor.ipynb").is_file()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Abre el notebook desde el proyecto o su carpeta de trabajo."
    )

ML_DIR = PROJECT_DIR / "ml"

In [4]:
ZIP_PATH = ML_DIR / "data/invoices_dataset.zip"
EXTRACTED_DIR = ML_DIR / "data/extracted"
REPORTS_DIR = ML_DIR / "outputs/reports"
LABELS_ENTRY = "labeled_invoices.csv"
KEY = ["source_file_path", "item_position"]

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"No se encuentra el paquete: {ZIP_PATH}")

dataset_archive_sha256 = sha256(ZIP_PATH.read_bytes()).hexdigest()

print(f"Paquete: {ZIP_PATH.name} · {ZIP_PATH.stat().st_size / 1_000_000:.1f} MB")

Paquete: invoices_dataset.zip · 10.5 MB


In [5]:
CATEGORIES = [
    "Alimentación",
    "Salud",
    "Educación, Arte y Cultura",
    "Vivienda",
    "Vestimenta",
    "Turismo Nacional",
    "Otros",
]

In [6]:
def normalize_entry(value: str) -> str:
    """Comprueba la ruta interna sin extraer archivos al disco."""
    path = PurePosixPath(value)

    if path.is_absolute() or ".." in path.parts or "\\" in value:
        raise ValueError("Ruta interna no admitida en el paquete.")

    return path.as_posix()


In [7]:
with ZipFile(ZIP_PATH) as archive:
    names = [entry.filename for entry in archive.infolist() if not entry.is_dir()]

    if len(names) != len(set(names)):
        raise ValueError("El ZIP contiene nombres de entrada repetidos.")

    xml_entries = sorted(
        normalize_entry(n) for n in names if n.lower().endswith(".xml")
    )

    labels = pd.read_csv(
        io.BytesIO(archive.read(LABELS_ENTRY)), dtype="string", keep_default_na=False
    )

print(f"Lectura del paquete: {len(xml_entries):,} XML y {len(labels):,} etiquetas.")

Lectura del paquete: 7,225 XML y 22,796 etiquetas.


## 2. Correspondencia de las etiquetas


Cada etiqueta corresponde a una línea específica de un comprobante y para identificarla se utiliza la ruta del archivo y la posición de la línea, comprobando después que la factura y la descripción coincidan. Si el mismo concepto aparece en otra compra, se conserva como un registro diferente, aunque tenga exactamente la misma descripción.

El CSV del ZIP conserva sus cabeceras externas `source_entry`, `access_key_id`, `detail_position` y `description`. Después de validarlas se traducen una sola vez a `source_file_path`, `anonymized_access_key`, `item_position` y `raw_description`; el resto de este flujo utiliza esos nombres comunes.


In [8]:
expected_labels = {
    "source_entry",
    "access_key_id",
    "detail_position",
    "description",
    "category",
}

if set(labels.columns) != expected_labels:
    raise ValueError(f"El CSV debe contener estas columnas: {sorted(expected_labels)}")

label_column_names = {
    "source_entry": "source_file_path",
    "access_key_id": "anonymized_access_key",
    "detail_position": "item_position",
    "description": "raw_description",
}

labels = labels.rename(columns=label_column_names)

labels["source_file_path"] = labels["source_file_path"].map(normalize_entry)

positions = pd.to_numeric(labels["item_position"], errors="coerce")

if positions.isna().any() or (positions < 1).any() or (positions % 1 != 0).any():
    raise ValueError("Las posiciones de etiquetas deben ser enteros desde 1.")

labels["item_position"] = positions.astype("int64")

if labels.duplicated(KEY).any():
    raise ValueError("Existen varias etiquetas para una misma línea de origen.")

if not labels["category"].isin(CATEGORIES).all():
    raise ValueError("Hay categorías vacías o ajenas al catálogo revisado.")

if not xml_entries or labels.empty:
    raise ValueError("El paquete debe contener facturas y etiquetas.")


## 3. Lectura del comprobante


In [9]:
def parse_xml(data: bytes | str):
    """Lee XML sin DTD, red ni resolución de entidades."""
    if isinstance(data, str):
        data = re.sub(r"^\ufeff?<\?xml[^?]*\?>", "", data).encode("utf-8")

    parser = etree.XMLParser(
        resolve_entities=False, no_network=True, load_dtd=False, recover=False
    )

    root = etree.fromstring(data, parser)

    if root.getroottree().docinfo.doctype or any(
        isinstance(n, etree._Entity) for n in root.iter()
    ):
        raise ValueError("El XML contiene una estructura no admitida.")

    return root


def read_invoice(data: bytes):
    """Obtiene el cuerpo de factura y conserva su huella canónica."""
    root = parse_xml(data)

    if etree.QName(root).localname == "autorizacion":
        embedded = root.xpath("./*[local-name()='comprobante']")

        if len(embedded) != 1 or not embedded[0].text or len(embedded[0]):
            raise ValueError("No hay una única factura embebida legible.")

        root = parse_xml(embedded[0].text)

    if etree.QName(root).localname != "factura":
        raise ValueError("El comprobante no es una factura.")

    body_hash = sha256(
        etree.tostring(root, method="c14n", with_comments=False)
    ).hexdigest()

    for node in root.iter():
        if isinstance(node.tag, str):
            node.tag = etree.QName(node).localname

    return root, body_hash

In [10]:
ANONYMIZED_VALUES = {
    "ruc": "[RUC_ANONIMIZADO]",
    "razonSocial": "[EMPRESA EMISORA ANONIMIZADA]",
    "nombreComercial": "[EMPRESA ANONIMIZADA]",
    "razonSocialComprador": "[COMPRADOR ANONIMIZADO]",
    "identificacionComprador": "[IDENTIFICACION_ANONIMIZADA]",
    "dirMatriz": "[DIRECCION ANONIMIZADA]",
    "dirEstablecimiento": "[DIRECCION ANONIMIZADA]",
}


def check_anonymization(invoice) -> None:
    """Verifica las máscaras del flujo de origen sin exportar identidades."""
    for node in invoice.iter():
        if node.tag == "Signature":
            raise ValueError("Hay una firma pendiente de retirar.")

        if (
            node.tag in ANONYMIZED_VALUES
            and (node.text or "").strip() != ANONYMIZED_VALUES[node.tag]
        ):
            raise ValueError(f"Falta anonimización en {node.tag}.")

        if (
            node.tag == "campoAdicional"
            and (node.text or "").strip() != "[VALOR_ANONIMIZADO]"
        ):
            raise ValueError("Hay información adicional pendiente de anonimizar.")

In [11]:
issues = []


def decimal_value(value: str, source: str, position: int | None, field: str):
    """Conserva un decimal finito o registra la incidencia sin inventar un cero."""
    try:
        number = Decimal(value)

        if not number.is_finite():
            raise InvalidOperation

        return number
    except (InvalidOperation, ValueError):
        issues.append(
            {
                "source_file_path": source,
                "item_position": position,
                "field": field,
                "reason": "missing_or_invalid_number",
            }
        )

        return None


def tax_total(nodes, source: str, position: int | None):
    """Calcula la suma de impuestos de una línea, o devuelve None si hay valores inválidos."""
    values = [
        decimal_value(n.findtext("valor", ""), source, position, "tax_amount")
        for n in nodes
    ]

    return (
        sum(values, Decimal(0))
        if values and all(v is not None for v in values)
        else None
    )

In [12]:
def invoice_row(invoice, source: str, data: bytes, body_hash: str) -> dict:
    """Extrae contexto de factura sin datos de emisor o comprador."""
    row = {
        "source_file_id": sha256(source.encode()).hexdigest()[:24],
        "source_file_path": source,
        "dataset_archive_sha256": dataset_archive_sha256,
        "anonymized_access_key": invoice.findtext("infoTributaria/claveAcceso"),
        "source_xml_sha256": sha256(data).hexdigest(),
        "invoice_body_sha256": body_hash,
        "invoice_version": invoice.get("version", ""),
        "issue_date_raw": invoice.findtext("infoFactura/fechaEmision", ""),
        "raw_currency": invoice.findtext("infoFactura/moneda", ""),
        "original_item_count": len(invoice.findall("detalles/detalle")),
    }

    fields = {
        "net_total": "totalSinImpuestos",
        "total_discount": "totalDescuento",
        "invoice_total": "importeTotal",
        "tip": "propina",
    }

    for field, tag in fields.items():
        value = invoice.findtext(f"infoFactura/{tag}", "")

        row[field + "_raw"] = value

        row[field] = (
            None
            if field == "tip" and not value
            else decimal_value(value, source, None, field)
        )

    row["tax_amount"] = tax_total(
        invoice.findall("infoFactura/totalConImpuestos/totalImpuesto"), source, None
    )

    return row

In [13]:
def detail_row(detail, invoice_data: dict, position: int) -> dict:
    """Extrae la línea y conserva sus valores para auditoría."""
    source = invoice_data["source_file_path"]

    row = {
        "source_file_id": invoice_data["source_file_id"],
        "source_file_path": source,
        "anonymized_access_key": invoice_data["anonymized_access_key"],
        "item_position": position,
        "raw_description": detail.findtext("descripcion", ""),
        "product_code": detail.findtext("codigoPrincipal", ""),
        "auxiliary_product_code": detail.findtext("codigoAuxiliar", ""),
        "additional_item_attributes": " | ".join(
            f"{node.get('nombre', '')}: {node.get('valor', '')}"
            for node in detail.findall("detallesAdicionales/detAdicional")
        ),
    }

    fields = {
        "quantity": "cantidad",
        "unit_price": "precioUnitario",
        "discount": "descuento",
        "line_total": "precioTotalSinImpuesto",
    }

    for field, tag in fields.items():
        value = detail.findtext(tag, "")

        row[field + "_raw"] = value
        row[field] = decimal_value(value, source, position, field)

    row["tax_amount"] = tax_total(
        detail.findall("impuestos/impuesto"), source, position
    )

    return row

## 4. Extracción de datos de la factura


In [14]:
invoice_rows, detail_rows, failures = [], [], []

with ZipFile(ZIP_PATH) as archive:
    for number, source in enumerate(xml_entries, start=1):
        try:
            data = archive.read(source)
            invoice, body_hash = read_invoice(data)

            check_anonymization(invoice)

            row = invoice_row(invoice, source, data, body_hash)

            invoice_rows.append(row)

            detail_rows.extend(
                detail_row(detail, row, position)
                for position, detail in enumerate(
                    invoice.findall("detalles/detalle"), start=1
                )
            )
        except (ValueError, etree.XMLSyntaxError, KeyError) as exc:
            failures.append({"source_file_path": source, "reason": str(exc)})

        if number % 1000 == 0 or number == len(xml_entries):
            print(f"Extracción: {number:,}/{len(xml_entries):,} facturas revisadas.")

invoices = pd.DataFrame(invoice_rows)
details = pd.DataFrame(detail_rows)

Extracción: 1,000/7,225 facturas revisadas.


Extracción: 2,000/7,225 facturas revisadas.


Extracción: 3,000/7,225 facturas revisadas.


Extracción: 4,000/7,225 facturas revisadas.


Extracción: 5,000/7,225 facturas revisadas.


Extracción: 6,000/7,225 facturas revisadas.


Extracción: 7,000/7,225 facturas revisadas.
Extracción: 7,225/7,225 facturas revisadas.


In [15]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def write_optional_report(frame: pd.DataFrame, name: str) -> None:
    """Genera un CSV de incidencias si hay registros"""
    path = REPORTS_DIR / name

    if frame.empty:
        path.unlink(missing_ok=True)
    else:
        frame.to_csv(path, index=False)

        print(f"Incidencias: {path.name} · {len(frame):,} registros.")

In [16]:
write_optional_report(pd.DataFrame(failures), "extraction_failures.csv")

if failures:
    raise ValueError(
        "Hay facturas no extraídas. Revisa extraction_failures.csv antes de continuar."
    )

if (
    details.empty
    or invoices["source_file_id"].duplicated().any()
    or details.duplicated(KEY).any()
):
    raise ValueError("No se ha obtenido una relación válida de facturas y conceptos.")

invoices["issue_date"] = pd.to_datetime(
    invoices["issue_date_raw"], format="%d/%m/%Y", errors="coerce"
)

In [17]:
coverage = details.merge(
    labels,
    on=KEY,
    how="outer",
    suffixes=("", "_label"),
    indicator=True,
    validate="one_to_one",
)

unmatched = coverage["_merge"].ne("both")

identity_mismatch = (
    coverage["anonymized_access_key"].ne(coverage["anonymized_access_key_label"]).fillna(True)
)

text_mismatch = coverage["raw_description"].ne(coverage["raw_description_label"]).fillna(True)

label_problems = coverage.loc[unmatched | identity_mismatch | text_mismatch, KEY].copy()
label_problems["reason"] = "label_xml_mismatch"

write_optional_report(label_problems, "label_alignment_issues.csv")

if not label_problems.empty:
    raise ValueError(
        "Hay diferencias entre etiquetas y XML. No se publicará una unión parcial."
    )

details = details.merge(labels[KEY + ["category"]], on=KEY, validate="one_to_one")

invalid_dates = invoices.loc[invoices["issue_date"].isna(), ["source_file_path"]].copy()
invalid_dates["field"], invalid_dates["reason"] = (
    "issue_date",
    "missing_or_invalid_date",
)
extraction_issues = pd.concat([pd.DataFrame(issues), invalid_dates], ignore_index=True)

write_optional_report(extraction_issues, "extraction_issues.csv")

print(f"Etiquetas integradas: {len(details):,}/{len(labels):,}")

Etiquetas integradas: 22,796/22,796


## 5. Archivos tabulares de salida


In [18]:
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

for name, frame in {"invoices": invoices, "details": details}.items():
    path = EXTRACTED_DIR / f"{name}.parquet"

    frame.to_parquet(path, index=False)

    restored = pd.read_parquet(path)

    if len(restored) != len(frame) or list(restored.columns) != list(frame.columns):
        raise RuntimeError(f"La lectura de vuelta no coincide: {name}.")

    print(f"Guardado {path.relative_to(PROJECT_DIR)} -  {len(frame):,} filas.")


Guardado ml/data/extracted/invoices.parquet -  7,225 filas.


Guardado ml/data/extracted/details.parquet -  22,796 filas.


In [19]:
display(
    pd.DataFrame(
        {
            "Tabla": ["Facturas de origen", "Conceptos etiquetados"],
            "Filas": [len(invoices), len(details)],
            "Columnas": [len(invoices.columns), len(details.columns)],
        }
    )
)

,Tabla,Filas,Columnas
0,Facturas de origen,7225,20
1,Conceptos etiquetados,22796,18


In [20]:
preview_columns = [
    "source_file_path",
    "item_position",
    "raw_description",
    "category",
    "quantity",
    "line_total",
]

display(details[preview_columns].head(8))


,source_file_path,item_position,raw_description,category,quantity,line_total
0,anonymized_invoices/doc_00000001.xml,1,Cargo de Reactivación por Mora,Otros,1,2.00
1,anonymized_invoices/doc_00000001.xml,2,PLAN CONEXION PAGO ANTICIPADO 12,Otros,1,12.00
2,anonymized_invoices/doc_00000002.xml,1,CLORO LEON 150 CC,Otros,22.000000,2.50
3,anonymized_invoices/doc_00000002.xml,2,GUITIG 3 LT,Alimentación,1.000000,1.27
4,anonymized_invoices/doc_00000002.xml,3,COLOMBINA WAFER UND,Alimentación,2.000000,0.09
5,anonymized_invoices/doc_00000003.xml,1,AGUA MINERAL C/GAS TA'RIKO 1.5,Alimentación,1,0.7
6,anonymized_invoices/doc_00000004.xml,1,GUITIG 3 LT,Alimentación,1.000000,1.27
7,anonymized_invoices/doc_00000004.xml,2,MICHU X LB,Alimentación,1.000000,0.89


## Archivos de salida


El archivo `ml/data/extracted/invoices.parquet` contiene una fila por cada factura procesada.

| Columna                                                               | Descripción                                                                                    |
| --------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------- |
| `source_file_id`                                                          | Identificador del archivo de origen generado a partir de su ruta dentro del ZIP.                       |
| `source_file_path`                                                        | Ruta del archivo dentro del ZIP.                                                               |
| `dataset_archive_sha256`                                                      | Huella del ZIP utilizado para relacionar los resultados de extracción, análisis y limpieza.    |
| `anonymized_access_key`                                                       | Clave de acceso ya anonimizada presente en el XML; puede repetirse entre copias.               |
| `source_xml_sha256`                                                      | Huella del archivo completo.                                                                   |
| `invoice_body_sha256`                                                      | Huella del contenido XML de la factura en formato canónico.                                    |
| `invoice_version`                                                     | Versión del formato XML de la factura.                                                         |
| `issue_date_raw`, `issue_date`                                        | Fecha original de la factura y fecha convertida a un formato utilizable.                       |
| `raw_currency`                                                            | Moneda indicada en la factura, sin realizar conversiones.                                      |
| `original_item_count`                                                        | Cantidad de conceptos incluidos en la factura.                                                 |
| `net_total`, `total_discount`, `invoice_total`, `tip`                 | Subtotal sin impuestos, descuentos, total de la factura y propina.                             |
| `net_total_raw`, `total_discount_raw`, `invoice_total_raw`, `tip_raw` | Valores de los importes.                |
| `tax_amount`                                                          | Suma de los impuestos indicados en la factura.


El archivo `ml/data/extracted/details.parquet` contiene una fila por cada concepto de las facturas. Cada línea se identifica mediante `source_file_path` y `item_position`, mientras que `source_file_id` permite relacionarla con su factura.

| Columna                                                            | Descripción                                                                                                           |
| ------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------- |
| `source_file_id`, `source_file_path`, `anonymized_access_key`                      | Identifican la factura de origen y el comprobante anonimizado al que pertenece cada concepto.                         |
| `item_position`                                                  | Posición original del concepto dentro de la factura, empezando desde 1. Se conserva para distinguir líneas similares. |
| `raw_description`                                                      | Descripción anonimizada del concepto, sin modificaciones adicionales durante la extracción.                           |
| `category`                                                         | Categoría asignada y revisada para ese concepto.                                                                      |
| `product_code`, `auxiliary_product_code`                                   | Códigos del producto tal como aparecen en la factura, incluidos los ceros iniciales.                                  |
| `additional_item_attributes`                                                  | Información adicional del concepto, almacenada por separado de su descripción.                                        |
| `quantity`, `unit_price`, `discount`, `line_total`                 | Cantidad, precio unitario, descuento e importe neto de cada línea.                                                    |
| `quantity_raw`, `unit_price_raw`, `discount_raw`, `line_total_raw` | Valores numéricos originales, conservados para revisar posibles errores de interpretación.                            |
| `tax_amount`                                                       | Suma de los impuestos declarados para ese concepto.                                                                   |

Los reportes se guardan en `ml/outputs/reports/` y solo se generan cuando se detectan problemas durante el procesamiento. 

- El archivo `extraction_failures.csv` registra las facturas que no pudieron leerse.
- El archivo `label_alignment_issues.csv` recoge las líneas que no coinciden con sus etiquetas.
- El archivo `extraction_issues.csv` señala los importes o fechas que no pudieron interpretarse correctamente.